# second-ear: continue the where-only training on Colab
Run the cells top to bottom (Runtime > Run all). Pick a T4 GPU first: Runtime > Change runtime type.

## 1. Check the GPU and connect Google Drive
Shows which graphics card Colab gave you, then asks permission to open your Drive.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

## 2. Settings
Where your files live on Drive, which code branch to use, and which models to train.

In [ ]:
DRIVE = '/content/drive/MyDrive/second-ear'
BRANCH = 'feature/colab-train'
MODELS = ['binaural', 'left', 'right']

## 3. Get the code and install it
Downloads the project from GitHub (or updates it) and installs it. Colab already has PyTorch, so we do not reinstall it. Colab's Python is newer than the project asks for, so we skip the version check and install the other libraries by hand.

In [ ]:
import os
os.environ['DRIVE'] = DRIVE
os.environ['BRANCH'] = BRANCH
if os.path.isdir('/content/second-ear/.git'):
    !git -C /content/second-ear fetch origin $BRANCH && git -C /content/second-ear checkout $BRANCH && git -C /content/second-ear pull origin $BRANCH
else:
    !git clone --branch $BRANCH https://github.com/JohnFok119/second-ear.git /content/second-ear
%cd /content/second-ear
!pip install -q --no-deps --ignore-requires-python -e .
!pip install -q numpy scipy soundfile librosa pyroomacoustics spaudiopy sofar matplotlib pyyaml tqdm requests "tensorboard>=2.21.0"
!python -c "import torch, secondear.train.heatmap; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 4. Unpack the features onto the Colab disk
Copies each float16 shard from Drive and unpacks it. A shard that was already unpacked is skipped, so re-running is quick.

In [ ]:
import glob, os, tarfile
DEST = '/content/second-ear/data/features/binaural_v2'
os.makedirs(DEST, exist_ok=True)
shards = sorted(glob.glob(f'{DRIVE}/features/feat_f16_*.tar'))
assert shards, f'no shards found in {DRIVE}/features'
for s in shards:
    marker = os.path.join(DEST, '.extracted_' + os.path.basename(s))
    if os.path.exists(marker):
        print('skip', os.path.basename(s))
        continue
    print('extract', os.path.basename(s))
    with tarfile.open(s) as t:
        t.extractall(DEST)
    open(marker, 'w').close()
print(len(glob.glob(DEST + '/train/*.npz')), 'train clips,', len(glob.glob(DEST + '/val/*.npz')), 'val clips')

## 5. Save progress to Drive
Makes the checkpoints and runs folders live on Drive, so a disconnect does not lose them. Your uploaded `where_binaural` files are picked up from here.

In [ ]:
import os, shutil
for name in ('checkpoints', 'runs'):
    target = f'{DRIVE}/{name}'
    link = f'/content/second-ear/{name}'
    os.makedirs(target, exist_ok=True)
    if os.path.islink(link):
        continue
    if os.path.isdir(link):
        shutil.rmtree(link)
    os.symlink(target, link)
print(os.listdir(f'{DRIVE}/checkpoints'))

## 6. Train
Trains each model in turn. A model with a `done.txt` is skipped; one with a `last.pt` carries on from where it stopped. After a disconnect, just run all cells again.

In [ ]:
import os, subprocess
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
for m in MODELS:
    ck = f'{DRIVE}/checkpoints/where_{m}'
    if os.path.exists(f'{ck}/done.txt'):
        print(m, ': already done, skipping')
        continue
    cmd = ['python', 'scripts/train_heatmap.py', '--config', f'configs/where_{m}.yaml', '--device', 'cuda']
    if os.path.exists(f'{ck}/last.pt'):
        cmd.append('--resume')
    print(m, ':', ' '.join(cmd), flush=True)
    log = f'{DRIVE}/runs/where_{m}_train.log'
    with open(log, 'a', encoding='utf-8') as fh:
        p = subprocess.Popen(cmd, cwd='/content/second-ear', stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='')
            fh.write(line)
            fh.flush()
        rc = p.wait()
    if rc != 0:
        raise RuntimeError(f'{m} training failed with exit code {rc}; see {log}')
    os.makedirs(ck, exist_ok=True)
    open(f'{ck}/done.txt', 'w').write('done\n')
print('all requested models finished')